# 📚 RAG en vivo: pregúntale a El Principito

En este notebook vamos a construir un sistema RAG paso a paso que permite hacerle preguntas al libro *El Principito* de Antoine de Saint-Exupéry.

**Lo que vamos a hacer:**
1. Instalar dependencias
2. Cargar el PDF
3. Dividir en chunks
4. Crear embeddings y guardar en ChromaDB
5. Hacer preguntas en vivo

> ⚠️ **Antes de empezar:** necesitas una API key de OpenRouter (gratis en [openrouter.ai](https://openrouter.ai))

## 0. Instalación de dependencias

Solo necesitas correr esta celda una vez.

In [ ]:
%pip install -r requirements.txt

## 1. Configuración

Ingresa tu API key de OpenRouter. 

In [2]:
import os
from getpass import getpass

# Ingresa tu API key (no se muestra en pantalla)
os.environ["OPENROUTER_API_KEY"] = getpass("🔑 OpenRouter API Key: ")

# Ruta al PDF — asegúrate de que el archivo esté en la misma carpeta que este notebook
PDF_PATH = "el_principito.pdf"

## 2. Cargar el PDF

Cargamos el PDF usando `PyPDFLoader`. Cada página del PDF se convierte en un objeto `Document` con su texto y metadatos (número de página, nombre del archivo).

In [3]:
import re
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(PDF_PATH)
docs = loader.load()

# Limpiar espacios y tabs extra que genera la extracción del PDF
for doc in docs:
    doc.page_content = re.sub(r'[ \t]+', ' ', doc.page_content)  # colapsa espacios/tabs
    doc.page_content = re.sub(r'\n{3,}', '\n\n', doc.page_content)  # colapsa saltos extra
    doc.page_content = doc.page_content.strip()

print(f"📄 Páginas cargadas: {len(docs)}")
print(f"\n--- Ejemplo: quinta página ---")
print(docs[5].page_content[:500])

c:\Users\lcgallardov\Documents\PCD\clase_rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📄 Páginas cargadas: 47

--- Ejemplo: quinta página ---
—Porque en mi tierra es todo tan pequeño…
Se inclinó hacia el dibujo y exclamó:
—¡Bueno, no tan pequeño…! Está dormido…
Y así fue como conocí al principito.
 
 
III
 
Me costó mucho tiempo comprender de dónde venía. El principito, que me
hacía muchas preguntas, jamás parecía oír las mías. Fueron palabras
pronunciadas al azar, las que poco a poco me revelaron todo. Así, cuando
distinguió por vez primera mi avión (no dibujaré mi avión, por tratarse de un
dibujo demasiado complicado para mí) me pre


## 3. Dividir en chunks

El PDF completo es demasiado largo para enviarlo al LLM de una sola vez. Lo dividimos en fragmentos pequeños (*chunks*) con un poco de solapamiento para no perder contexto entre uno y otro.

In [26]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # caracteres por chunk
    chunk_overlap=150,    # solapamiento entre chunks
)
chunks = splitter.split_documents(docs)

print(f"✂️  Total de chunks: {len(chunks)}")
print(f"\n--- Ejemplo: chunk #10 ---")
print(chunks[10].page_content)
print(f"\nMetadatos: {chunks[10].metadata}")

✂️  Total de chunks: 231

--- Ejemplo: chunk #10 ---
Viví así, solo, nadie con quien poder hablar verdaderamente, hasta cuando
hace seis años tuve una avería en el desierto de Sahara. Algo se había
estropeado en el motor. Como no llevaba conmigo ni mecánico ni pasajero
alguno, me dispuse a realizar, yo solo, una reparación difícil. Era para mí una
cuestión de vida o muerte, pues apenas tenía agua de beber para ocho días.
La primera noche me dormí sobre la arena, a unas mil millas de distancia

Metadatos: {'producer': 'calibre 2.53.0 [http://calibre-ebook.com]', 'creator': 'calibre 2.53.0 [http://calibre-ebook.com]', 'creationdate': '2018-02-27T10:18:33+00:00', 'author': 'Antoine De Saint-Exupéry', 'title': 'El Principito', 'source': 'el_principito.pdf', 'total_pages': 47, 'page': 3, 'page_label': '4'}


## 4. Crear embeddings y guardar en ChromaDB

Convertimos cada chunk a un vector numérico usando un modelo de embeddings. Estos vectores se guardan en ChromaDB, una base de datos vectorial local.

> ⏳ Esta celda puede tardar 1-2 minutos la primera vez — está descargando el modelo de embeddings.

In [27]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

print("⏳ Creando embeddings... (puede tardar un momento)")

embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",  # multilingüe, funciona bien en español
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_principito",  # se guarda localmente
)

print(f"✅ ChromaDB listo con {vectorstore._collection.count()} vectores")

⏳ Creando embeddings... (puede tardar un momento)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5920.04it/s]


✅ ChromaDB listo con 810 vectores


## 5. Crear el retriever y el LLM

El retriever busca los chunks más similares a una pregunta. El LLM recibe esos chunks como contexto y genera la respuesta.

In [40]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Retriever: recupera los 3 chunks más relevantes
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5,
                   "fetch_k": 20,
                   "lambda_mult": 0.7},
)

ejemplo = retriever.invoke("¿De qué planeta es el principito?")  
for c in ejemplo:
    print(f"Chunk: \n{c.page_content[:200]}...")

Chunk: 
descubre uno de estos planetas, le da por nombre un número. Le llama, por
ejemplo, "el asteroide 3251".
Tengo poderosas razones para creer que el planeta del cual venía el
principito era el asteroide ...
Chunk: 
Entonces el principito señaló con gravedad:
—¡No importa, es tan pequeña mi tierra!
Y agregó, quizás, con un poco de melancolía:
—Derecho, camino adelante… no se puede ir muy lejos.
 
 
IV
 
De esta m...
Chunk: 
—¡Buenos días! —exclamó el principito al acaso.
—¡Buenos días! ¡Buenos días! ¡Buenos días! —respondió el eco.
—¿Quién eres tú? —preguntó el principito.
—¿Quién eres tú?... ¿Quién eres tú?... ¿Quién er...
Chunk: 
—Sobre la Tierra, en África —respondió la serpiente.
—¡Ah! ¿Y no hay nadie sobre la Tierra?
—Esto es el desierto. En los desiertos no hay nadie. La Tierra es muy
grande —dijo la serpiente.
El principi...
Chunk: 
el principito durante su viaje.
 
 
XII
 
El tercer planeta estaba habitado por un bebedor. Fue una visita muy corta,...


In [38]:
# LLM via OpenRouter
llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    model="openai/gpt-oss-20b:free",
    temperature=0.0,
)

# Prompt: instruye al LLM a responder SOLO con el contexto
prompt = PromptTemplate.from_template("""
Eres un asistente experto en el libro El Principito de Antoine de Saint-Exupéry.
Responde la pregunta usando la siguiente información del libro.
Si la respuesta no está en el contexto, di "No encontré esa información en el texto."

Contexto:
{context}

Pregunta: {question}

Respuesta:
""")

# Cadena LCEL
chain = (
    {"question": RunnablePassthrough(),
     "context": retriever}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ Pipeline RAG listo")

✅ Pipeline RAG listo


In [30]:
chain.invoke("¿De qué planeta es el principito?")

'El principito es originario del asteroide **B612**.'

## 6. ¡A preguntar! ✅ Preguntas que funcionan bien

Estas preguntas tienen respuesta clara en el texto.

In [31]:
def preguntar(pregunta):
    """Hace una pregunta al pipeline RAG y muestra la respuesta con los chunks recuperados."""
    print(f"❓ Pregunta: {pregunta}")
    print("-" * 60)
    
    # Ver qué chunks recuperó el retriever
    chunks_recuperados = retriever.invoke(pregunta)
    print("📎 Chunks recuperados:")
    for i, chunk in enumerate(chunks_recuperados, 1):
        print(f"  [{i}] Página {chunk.metadata.get('page', '?')}: {chunk.page_content[:120]}...")
    
    print()
    respuesta = chain.invoke(pregunta)
    print(f"🤖 Respuesta:\n{respuesta}")
    print("=" * 60)

In [32]:
preguntar("¿Qué le pidió el principito al piloto que dibujara?")

❓ Pregunta: ¿Qué le pidió el principito al piloto que dibujara?
------------------------------------------------------------
📎 Chunks recuperados:
  [1] Página 5: dibujo demasiado complicado para mí) me preguntó:
—¿Qué cosa es esa? —Eso no es una cosa. Eso vuela. Es un avión, mi
avi...
  [2] Página 5: —Porque en mi tierra es todo tan pequeño…
Se inclinó hacia el dibujo y exclamó:
—¡Bueno, no tan pequeño…! Está dormido…
...
  [3] Página 39: que nuevamente se había sentado junto a mí.
—¿Qué promesa?
—Ya sabes... el bozal para mi cordero... soy responsable de m...
  [4] Página 30: —¡Buenos días! —exclamó el principito al acaso.
—¡Buenos días! ¡Buenos días! ¡Buenos días! —respondió el eco.
—¿Quién er...
  [5] Página 39: Y volvió a reír.
—Eres injusto, muchachito; yo no sabía dibujar más que boas cerradas y
boas abiertas.
—¡Oh, todo se arr...

🤖 Respuesta:
El principito le pidió al piloto que dibujara un cordero.


In [34]:
preguntar("¿Qué significa domesticar según el zorro?")

❓ Pregunta: ¿Qué significa domesticar según el zorro?
------------------------------------------------------------
📎 Chunks recuperados:
  [1] Página 32: —Estoy aquí, bajo el manzano —dijo la voz.
—¿Quién eres tú? —preguntó el principito—. ¡Qué bonito eres!
—Soy un zorro —d...
  [2] Página 32: "domesticar"?
—Los hombres —dijo el zorro— tienen escopetas y cazan. ¡Es muy
molesto! Pero también crían gallinas. Es lo...
  [3] Página 32: —volvió a preguntar el principito.
—Es una cosa ya olvidada —dijo el zorro—, significa "crear vínculos... "
—¿Crear vínc...
  [4] Página 34: parecerían y yo no tendría vacaciones.
De esta manera el principito domesticó al zorro. Y cuando se fue
acercando el día...
  [5] Página 32: ella me ha domesticado...
—Es posible —concedió el zorro—, en la Tierra se ven todo tipo de cosas.
—¡Oh, no es en la Tie...

🤖 Respuesta:
El zorro explica que “domesticar” significa “crear vínculos…”.


In [36]:
preguntar("¿Qué significado tiene la frase lo esencial es invisible a los ojos?")

❓ Pregunta: ¿Qué significado tiene la frase lo esencial es invisible a los ojos?
------------------------------------------------------------
📎 Chunks recuperados:
  [1] Página 35: —Adiós —dijo el zorro—. He aquí mi secreto, que no puede ser más
simple: sólo con el corazón se puede ver bien; lo esenc...
  [2] Página 38: más importante es invisible... "
Como sus labios entreabiertos esbozaron una sonrisa, me dije: "Lo que más
me emociona d...
  [3] Página 32: —volvió a preguntar el principito.
—Es una cosa ya olvidada —dijo el zorro—, significa "crear vínculos... "
—¿Crear vínc...
  [4] Página 12: las sumas de un señor gordo y colorado? Y si yo sé de una flor única en el
mundo y que no existe en ninguna parte más qu...
  [5] Página 42: —Lo más importante nunca se ve...
—Indudablemente...
—Es lo mismo que la flor. Si te gusta una flor que habita en una es...

🤖 Respuesta:
La frase “lo esencial es invisible a los ojos” significa que lo más importante y verdadero de las cosas no se percibe

In [39]:
preguntar("¿Qué hacía el rey en su planeta?")

❓ Pregunta: ¿Qué hacía el rey en su planeta?
------------------------------------------------------------
📎 Chunks recuperados:
  [1] Página 16: —Señor. . . ¿sobre qué ejerce su poder?
—Sobre todo —contestó el rey con gran ingenuidad.
—¿Sobre todo?
El rey, con un g...
  [2] Página 27: espinas para defenderse contra el mundo. ¡Y la he dejado allá sola en mi
casa!". Por primera vez se arrepintió de haber ...
  [3] Página 17: necesidad de arrastrar su silla. Y como se sentía un poco triste al recordar su
pequeño planeta abandonado, se atrevió a...
  [4] Página 19: el principito durante su viaje.
 
 
XII
 
El tercer planeta estaba habitado por un bebedor. Fue una visita muy corta,...
  [5] Página 16: se transformara en ave marina y el general no me obedeciese, la culpa no sería
del general, sino mía".
—¿Puedo sentarme?...

🤖 Respuesta:
El rey se dedica a gobernar su planeta, proclamando su autoridad sobre él, los demás planetas y las estrellas, y emitiendo órdenes (por ejemplo, ordena que 

## 7. ❌ Pregunta que el RAG NO puede responder

Una de las ventajas del RAG es que **sabe cuándo no sabe** — si el contexto no tiene la respuesta, debería decirlo.

In [11]:
# ❌ Pregunta fuera del libro — el modelo debería responder que no tiene esa info
preguntar("¿En qué año nació Antoine de Saint-Exupéry?")

❓ Pregunta: ¿En qué año nació Antoine de Saint-Exupéry?
------------------------------------------------------------
📎 Chunks recuperados:
  [1] Página 0: El Principito
 
Por
 
Antoine De Saint-Exupéry...
  [2] Página 23: "Este hombre, quizás, es absurdo. Sin embargo, es menos absurdo que el
rey, el vanidoso, el hombre de negocios y el bebe...
  [3] Página 27: —De las flores no tomamos nota.
—¿Por qué? ¡Son lo más bonito!
—Porque las flores son efímeras.
—¿Qué significa "efímera...

🤖 Respuesta:
No encontré esa información en el texto.


In [12]:
# ❌ Pregunta ambigua que puede confundir al retriever
preguntar("¿Qué pasó al final?")

❓ Pregunta: ¿Qué pasó al final?
------------------------------------------------------------
📎 Chunks recuperados:
  [1] Página 12: salir de allí una aparición milagrosa; pero la flor no acababa de preparar su...
  [2] Página 41: muere a tiros de carabina.
—Me alegra —dijo el principito— que hayas encontrado lo que faltaba a
tu máquina. Así podrás ...
  [3] Página 44: Me senté, ya no podía mantenerme en pie.
—Ahí está... eso es todo...
Vaciló todavía un instante, luego se levantó y dio ...

🤖 Respuesta:
No encontré esa información en el texto.


## 8. 🔍 Pregunta libre

¡Escribe tu propia pregunta!

In [ ]:
mi_pregunta = input("✏️  Escribe tu pregunta: ")
preguntar(mi_pregunta)